# Load pose-tracking `.h5` files

This notebook loads DeepLabCut (DLC) pose-tracking `.h5` files from the local `body-kinematics/pose_data` directory.

Each `.h5` file stores a pandas `DataFrame` (PyTables format, key `df_with_missing`) with a 3-level column `MultiIndex`:
`scorer / bodyparts / coords`, where `coords` is one of `x`, `y`, `likelihood`.

The loading/reshaping helpers below are written as standalone functions so they can be lifted into the
`pirouette_data` package later (e.g. `pirouette_data/io/pose.py`).

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

## Configuration

Point `POSE_DIR` at the folder holding the `.h5` files.

In [ ]:
POSE_DIR = Path(r"C:/Users/brandon.pratt/Desktop/data/body-kinematics/pose_data")

assert POSE_DIR.exists(), f"Pose directory not found: {POSE_DIR}"

h5_files = sorted(POSE_DIR.glob("*.h5"))
print(f"Found {len(h5_files)} .h5 file(s):")
for f in h5_files:
    print("  ", f.name)

## Loader helpers (package-ready)

In [ ]:
def load_pose_h5(path: str | Path) -> pd.DataFrame:
    """Load a single DeepLabCut pose `.h5` file.

    Args:
        path: Path to a DLC `.h5` file (pandas/PyTables format).

    Returns:
        A `DataFrame` whose columns are a 3-level MultiIndex
        (``scorer``, ``bodyparts``, ``coords``) and whose index is the frame number.
    """
    df = pd.read_hdf(path)
    df.index.name = "frame"
    return df


def get_bodyparts(df: pd.DataFrame) -> list[str]:
    """Return the list of tracked body-part names in a DLC pose DataFrame.

    Args:
        df: A DLC pose DataFrame as returned by :func:`load_pose_h5`.

    Returns:
        Ordered, de-duplicated list of body-part names.
    """
    return list(dict.fromkeys(df.columns.get_level_values("bodyparts")))


def drop_scorer_level(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the (single) DLC ``scorer`` column level for easier indexing.

    Args:
        df: A DLC pose DataFrame with a ``scorer/bodyparts/coords`` column MultiIndex.

    Returns:
        DataFrame with a 2-level ``bodyparts/coords`` column MultiIndex.
    """
    return df.droplevel("scorer", axis=1)


def pose_to_long(df: pd.DataFrame) -> pd.DataFrame:
    """Reshape a wide DLC pose DataFrame into a tidy/long DataFrame.

    Args:
        df: A DLC pose DataFrame as returned by :func:`load_pose_h5`.

    Returns:
        Long DataFrame with columns ``frame``, ``bodypart``, ``x``, ``y``, ``likelihood``.
    """
    flat = drop_scorer_level(df)
    long = (
        flat.stack(level="bodyparts", future_stack=True)
        .rename_axis(index=["frame", "bodypart"])
        .reset_index()
    )
    return long

## Load one file and inspect

In [ ]:
df = load_pose_h5(h5_files[0])

print(f"File:       {h5_files[0].name}")
print(f"Shape:      {df.shape}  (frames x [bodyparts*3])")
print(f"Scorer:     {df.columns.get_level_values('scorer')[0]}")
print(f"Bodyparts:  {get_bodyparts(df)}")
df.head()

In [ ]:
# Tidy/long form is often easier to filter, group, and plot
long = pose_to_long(df)
long.head()

## Quick sanity plot

Plot the x/y trajectory of one body part for the first N frames.

In [ ]:
bodyparts = get_bodyparts(df)
bp = bodyparts[-1]  # e.g. an animal body part rather than a chamber corner
n_frames = 5000

flat = drop_scorer_level(df)
sub = flat[bp].iloc[:n_frames]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sub.index, sub["x"], label="x", lw=0.8)
axes[0].plot(sub.index, sub["y"], label="y", lw=0.8)
axes[0].set(xlabel="frame", ylabel="pixel", title=f"{bp}: x/y vs frame")
axes[0].legend()

sc = axes[1].scatter(sub["x"], sub["y"], c=sub["likelihood"], s=3, cmap="viridis")
axes[1].set(xlabel="x (px)", ylabel="y (px)", title=f"{bp}: trajectory")
axes[1].invert_yaxis()  # image coordinates: origin top-left
fig.colorbar(sc, ax=axes[1], label="likelihood")
plt.tight_layout()
plt.show()

## Load all files

Concatenate every `.h5` in the directory into a single DataFrame, tagging each row with its source file.
The DLC frame index restarts at 0 per file, so we keep the filename as an identifier.

In [ ]:
def load_pose_dir(pose_dir: str | Path) -> dict[str, pd.DataFrame]:
    """Load every DLC pose `.h5` file in a directory.

    Args:
        pose_dir: Directory containing DLC `.h5` files.

    Returns:
        Mapping of filename (stem) -> pose DataFrame.
    """
    pose_dir = Path(pose_dir)
    return {p.stem: load_pose_h5(p) for p in sorted(pose_dir.glob("*.h5"))}


pose_by_file = load_pose_dir(POSE_DIR)

summary = pd.DataFrame(
    {
        "file": list(pose_by_file),
        "n_frames": [d.shape[0] for d in pose_by_file.values()],
        "n_columns": [d.shape[1] for d in pose_by_file.values()],
    }
)
summary